# Spot prices

Download daily spot prices from "Hva koster strømmen".

In [1]:
import sys
from pathlib import Path

project_root = next(
    parent
    for candidate in [Path.cwd(), *Path.cwd().parents]
    for parent in [candidate]
    if (parent / "bronze_layer" / "spot_prices.py").exists()
)
sys.path.insert(0, str(project_root))

from bronze_layer.spot_prices import ingest

/home/roland/UiS/rewrite/group14/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Small test

2 days and 2 price areas. This makes 4 requests.

In [2]:
test_result = ingest(
    start="2024-01-01",
    end="2024-01-03",
    areas=["NO1", "NO2"],
)

print({k: v for k, v in test_result.items() if k != "failures"})
print("Failures:", test_result["failures"][:5])

Spot prices: 100%|██████████| 4/4 [00:00<00:00, 35.15day-area/s, cached=0, downloaded=4, failed=0]

{'run_id': '5ce87377d8074f56b979bef50d7c3459', 'total_requests': 4, 'downloaded_count': 4, 'cached_count': 0, 'failed_count': 0}
Failures: []


## Inspect one saved raw response


In [3]:
import json

raw_file = (
    project_root / "data/bronze/spot_prices/area=NO1/date=2024-01-01/response.json"
)
if raw_file.exists():
    rows = json.loads(raw_file.read_text(encoding="utf-8"))
    print("Intervals:", len(rows))
    print(json.dumps(rows[:2], indent=2, ensure_ascii=False))
else:
    print("Run the small test first.")

Intervals: 24
[
  {
    "NOK_per_kWh": 0.60463,
    "EUR_per_kWh": 0.05379,
    "EXR": 11.2405,
    "time_start": "2024-01-01T00:00:00+01:00",
    "time_end": "2024-01-01T01:00:00+01:00"
  },
  {
    "NOK_per_kWh": 0.55056,
    "EUR_per_kWh": 0.04898,
    "EXR": 11.2405,
    "time_start": "2024-01-01T01:00:00+01:00",
    "time_end": "2024-01-01T02:00:00+01:00"
  }
]


## 4. Full historical run

Previously downloaded days are reused unless `refresh=True` is set.

In [4]:
full_result = ingest(
    start="2024-01-01",
    end="2026-01-01",
)
print({k: v for k, v in full_result.items() if k != "failures"})
print("First failures:", full_result["failures"][:10])

Spot prices: 100%|██████████| 3655/3655 [01:14<00:00, 49.21day-area/s, cached=4, downloaded=3651, failed=0]

{'run_id': 'c20cc272791c4319b1a5a526ee8ced87', 'total_requests': 3655, 'downloaded_count': 3651, 'cached_count': 4, 'failed_count': 0}
First failures: []
